# Lab Filter Audit & Prototype

**Goals:**
1. Audit what the ctproc lab extractor finds on **topic text** (patient measurements) vs **criteria text** (trial thresholds)
2. Build an inline ground-truth test set for both parsers
3. Prototype `extract_lab_constraints` — a new function that *preserves* operator direction from criteria text
4. Demonstrate **soft scoring** with tunable smoothing parameter α

**Key distinction:**
- Topic text: *measurement* — `"Hgb 8.0 g/dL"` → patient has this value
- Criteria text: *constraint* — `"Hemoglobin ≥ 10 g/dL"` → trial requires value above threshold

The existing `extract_lab_values` strips comparators as filler (correct for measurements, wrong for constraints).

**Scoring:**  
Hard filtering has the same TREC qrel incompatibility as the demographic filter — a trial requiring Hgb ≥ 10
can be labeled `relevant` for an Hgb=8 patient if the condition matches topically.
Instead, use a **soft compatibility score** as a reranking signal:

```
score(topic, trial) = (constraints_satisfied + α) / (evaluable + α)
```

where `evaluable` = analytes present in *both* topic labs and trial constraints.
When evaluable=0 (no shared analytes): score = α/α = 1.0 (neutral, no info → don't penalize).

In [ ]:
# ── Colab setup — run once, then restart runtime if prompted ────────────
# Skip this cell if running locally.

# Data files live in the ctmatch repo, so clone it for the data.
# The pip install gets the package (and ctproc as a dependency).
!git clone -q https://github.com/semajyllek/ctmatch.git /content/ctmatch
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q sympy==1.13.1  # pin for torch compat

In [ ]:
import os

# ── data paths ──────────────────────────────────────────────────────────
# Colab: data comes from the cloned repo (setup cell above)
DATA_ROOT = '/content/ctmatch/data'

# Local override (running outside Colab):
# DATA_ROOT = '../data'

TREC21_TOPICS = os.path.join(DATA_ROOT, 'trec_data/processed_trec_data/processed_trec21_topics.jsonl')
KZ_TOPICS     = os.path.join(DATA_ROOT, 'kz_data/processed_kz_data/processed_kz_topics.jsonl')
KZ_DATA       = os.path.join(DATA_ROOT, 'kz_data/kz_data.jsonl')

print('Data paths:')
for p in [TREC21_TOPICS, KZ_TOPICS, KZ_DATA]:
    print(f"  {'✓' if os.path.exists(p) else '✗'} {p}")

In [ ]:
import re
import json
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import Optional, List

import pandas as pd

from ctproc.lab.extractor import extract_lab_values
from ctproc.lab.patterns import ANALYTE_PATTERN, parse_number, parse_unit
from ctproc.lab.reference_ranges import get_lab_test
from ctproc.lab.types import LabValue

print('imports ok')

---
## Section 1 — Topic-side audit

Run `extract_lab_values` on all TREC21 and KZ topic texts.
Topics are patient descriptions — we expect measurement extractions like `Hgb 8.0 g/dL`.

In [ ]:
def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f]

trec21 = load_jsonl(TREC21_TOPICS)
kz     = load_jsonl(KZ_TOPICS)
print(f'TREC21: {len(trec21)} topics')
print(f'KZ:     {len(kz)} topics')

In [ ]:
def audit_topics(topics, label):
    hits, analyte_counter, examples = 0, Counter(), []
    for t in topics:
        text = t.get('raw_text', '')
        vals = extract_lab_values(text)
        if vals:
            hits += 1
            for v in vals:
                analyte_counter[v.analyte] += 1
            if len(examples) < 5:
                examples.append((t.get('id', '?'), [(v.analyte, v.value, v.unit) for v in vals[:4]]))
    print(f'\n{label}: {hits}/{len(topics)} topics ({100*hits/len(topics):.0f}%) have extractable lab values')
    print('Top analytes:')
    for analyte, cnt in analyte_counter.most_common(10):
        print(f'  {analyte}: {cnt}')
    print('Sample extractions:')
    for tid, vals in examples:
        print(f'  Topic {tid}: {vals}')
    return analyte_counter

trec21_analytes = audit_topics(trec21, 'TREC21')
kz_analytes     = audit_topics(kz,     'KZ')

---
## Section 2 — Criteria-side audit

Run the same extractor on trial criteria text (the `doc` field in `kz_data.jsonl`).
Criteria express *thresholds*, e.g. `"serum creatinine > 2.0 mg/dL"`.
The extractor will find the numeric value but **silently drop the operator** —
we get `Creatinine=2.0` with no `>`. This cell measures parse rate and shows the operator loss.

In [ ]:
kz_raw = load_jsonl(KZ_DATA)

criteria_hits, criteria_analytes = 0, Counter()
operator_loss_examples = []

_OP_NEARBY = re.compile(r'(?:≥|>=|≤|<=|>|<|at\s+least|no\s+more\s+than|greater\s+than|less\s+than)', re.I)

for d in kz_raw:
    doc = d.get('doc', '')
    vals = extract_lab_values(doc)
    if vals:
        criteria_hits += 1
        for v in vals:
            criteria_analytes[v.analyte] += 1
            snippet = doc[max(0, v.start):v.end + 30]
            if _OP_NEARBY.search(snippet) and len(operator_loss_examples) < 8:
                operator_loss_examples.append((v.analyte, v.value, v.unit, snippet.strip()))

print(f'Criteria docs: {criteria_hits}/{len(kz_raw)} ({100*criteria_hits/len(kz_raw):.0f}%) have extractable lab values')
print('Top analytes in criteria:')
for a, c in criteria_analytes.most_common(10):
    print(f'  {a}: {c}')
print('\nOperator loss examples (operator present in raw text but dropped by extractor):')
for analyte, val, unit, snippet in operator_loss_examples:
    print(f'  extracted: {analyte}={val}{unit}')
    print(f'  raw text:  {snippet[:80]}')
    print()

---
## Section 3 — Ground truth test set

Inline annotated examples for both parsers.
These become the regression test suite for the constraint extractor.

Format: `(input_text, expected_analyte, expected_op, expected_threshold, expected_unit)`

In [ ]:
# Constraint test cases: criteria text → LabConstraint
CONSTRAINT_CASES = [
    # Symbol operators
    ('Hemoglobin >= 10 g/dL',              'Hemoglobin',               '>=', 10.0,     'g/dL'),
    ('Hgb ≥ 8.0 g/dL',                    'Hemoglobin',               '>=', 8.0,      'g/dL'),
    ('serum creatinine > 2.0 mg/dL',       'Creatinine',               '>',  2.0,      'mg/dL'),
    ('creatinine >3.0 mg/dl',              'Creatinine',               '>',  3.0,      'mg/dL'),
    ('WBC >= 4,000 /mm3',                  'White blood cells',         '>=', 4000.0,   '/mm3'),
    ('ANC >=1,000 /mm3',                   'Absolute neutrophil count', '>=', 1000.0,   '/mm3'),
    ('Platelet count < 100,000 /mm3',      'Platelet count',           '<',  100000.0,  '/mm3'),
    ('ALT <= 2.5 x ULN',                   'ALT',                      '<=', 2.5,      'x ULN'),
    ('AST ≤ 3 x ULN',                      'AST',                      '<=', 3.0,      'x ULN'),
    ('eGFR > 60 mL/min',                   'eGFR',                     '>',  60.0,     'mL/min'),
    ('BUN > 40',                           'BUN',                      '>',  40.0,     'mg/dL'),
    # Word-based operators
    ('Hemoglobin at least 9 g/dL',         'Hemoglobin',               '>=', 9.0,      'g/dL'),
    ('creatinine no more than 1.5 mg/dL',  'Creatinine',               '<=', 1.5,      'mg/dL'),
    ('platelet count greater than 50,000', 'Platelet count',           '>',  50000.0,  '/mm3'),
    ('Hgb less than 7 g/dL',               'Hemoglobin',               '<',  7.0,      'g/dL'),
    ('INR no greater than 1.5',            'INR',                      '<=', 1.5,      ''),
    ('HbA1c greater than or equal to 7.5%','Hemoglobin A1c',           '>=', 7.5,      '%'),
    # Colon / label style
    ('Hemoglobin (Hgb): >= 9.0 g/dL',     'Hemoglobin',               '>=', 9.0,      'g/dL'),
    # No explicit operator
    ('Hemoglobin A1c 7.5%',               'Hemoglobin A1c',            '=',  7.5,      '%'),
]

# Measurement test cases: topic text → LabValue
MEASUREMENT_CASES = [
    ('Hgb 8.0 g/dL',             'Hemoglobin',           8.0,      'g/dL'),
    ('creatinine 1.4 mg/dL',     'Creatinine',           1.4,      'mg/dL'),
    ('Platelet count 300,000 /mm3','Platelet count',      300000.0, '/mm3'),
    ('HbA1c 6.5%',               'Hemoglobin A1c',       6.5,      '%'),
    ('WBC 8,000 /mm3',           'White blood cells',    8000.0,   '/mm3'),
    ('TSH 2.35 mU/L',            'TSH',                  2.35,     'mU/L'),
    ('Hemoglobin 13 g/dl',       'Hemoglobin',           13.0,     'g/dl'),
    ('eGFR 60 mL/min',           'eGFR',                 60.0,     'mL/min'),
]

print(f'{len(CONSTRAINT_CASES)} constraint test cases')
print(f'{len(MEASUREMENT_CASES)} measurement test cases')

In [ ]:
# Verify measurement cases against existing extractor
passes, fails = 0, []
for text, exp_analyte, exp_val, exp_unit in MEASUREMENT_CASES:
    vals = extract_lab_values(text)
    if not vals:
        fails.append(('NO_EXTRACT', text, exp_analyte, exp_val))
        continue
    v = vals[0]
    if v.analyte != exp_analyte or abs(v.value - exp_val) > 1e-6:
        fails.append(('WRONG', text, f'{v.analyte}={v.value}', f'expected {exp_analyte}={exp_val}'))
    else:
        passes += 1

print(f'Measurement extractor: {passes}/{len(MEASUREMENT_CASES)} pass')
for f in fails:
    print(f'  FAIL {f}')

---
## Section 4 — Constraint extractor prototype

`extract_lab_constraints(text)` — like the existing extractor but **captures the operator**.

Prototyped inline here so we can validate against the ground truth set before promoting to ctproc.

In [ ]:
@dataclass
class LabConstraint:
    """A threshold constraint extracted from clinical trial eligibility criteria text."""
    analyte: str
    op: str           # '<', '<=', '>', '>=', '='
    threshold: float
    threshold_high: Optional[float] = None  # populated for range constraints "9-12"
    unit: str = ""
    start: int = 0
    end: int = 0
    raw_text: str = ""

    def satisfied_by(self, patient_value: float) -> bool:
        ops = {'<': lambda a, b: a < b, '<=': lambda a, b: a <= b,
               '>': lambda a, b: a > b, '>=': lambda a, b: a >= b,
               '=': lambda a, b: abs(a - b) < 1e-9}
        if self.threshold_high is not None:
            return self.threshold <= patient_value <= self.threshold_high
        return ops[self.op](patient_value, self.threshold)


# Captures a symbol comparator after optional (ABBREV) and whitespace/colon
_SYM_OP = re.compile(
    r'^(?:\([A-Z]{2,6}\)\s*)?'
    r'[\s:]*'
    r'(≥|>=|≤|<=|>|<|=)?\s*',
    re.IGNORECASE,
)

# Word-based directional phrases, longest match first
_WORD_OPS = [
    (re.compile(r'^(?:at\s+least|no\s+less\s+than|greater\s+than\s+or\s+equal\s+to)[\s:]*', re.I), '>='),
    (re.compile(r'^(?:no\s+(?:greater|more)\s+than|less\s+than\s+or\s+equal\s+to)[\s:]*',   re.I), '<='),
    (re.compile(r'^(?:greater\s+than|above|over)[\s:]*',                                     re.I), '>'),
    (re.compile(r'^(?:less\s+than|below|under)[\s:]*',                                       re.I), '<'),
    (re.compile(r'^(?:equal\s+to|approximately)[\s:]*',                                      re.I), '='),
    (re.compile(r'^(?:count|level|of|is|was|at)[\s:]*',                                      re.I), None),  # noise
]


def _parse_comparator(text: str) -> tuple:
    """Return (op, remaining_text). op defaults to '=' when no comparator is found."""
    m = _SYM_OP.match(text)
    if m:
        sym  = m.group(1)
        rest = text[m.end():]
        if sym:
            return sym.replace('≥', '>=').replace('≤', '<='), rest
        text = rest
    for pattern, op in _WORD_OPS:
        wm = pattern.match(text)
        if wm:
            text = text[wm.end():]
            if op is not None:
                return op, text
            return _parse_comparator(text)  # noise word — keep scanning
    return '=', text


def extract_lab_constraints(text: str) -> List[LabConstraint]:
    """
    Extract lab threshold constraints from eligibility criteria text.

    Unlike extract_lab_values, this preserves the comparator operator so
    constraints can be compared against patient measurements.
    """
    results = []
    for m in ANALYTE_PATTERN.finditer(text):
        lab_test = get_lab_test(m.group(1))
        if lab_test is None:
            continue

        after = text[m.end(): m.end() + 150]
        op, cleaned = _parse_comparator(after.strip())
        num = parse_number(cleaned)
        if num is None:
            continue
        val, rest = num

        # ULN pattern: "2.5 x ULN" / "2.5 times upper limit of normal"
        uln_m = re.match(
            r'\s*(?:x\s+|times?\s+)?(?:the\s+)?(?:upper\s+limit\s+of\s+normal|ULN|institutional\s+normal)',
            rest, re.IGNORECASE,
        )
        if uln_m:
            end_pos = m.end() + len(after) - len(cleaned) + len(str(val)) + uln_m.end()
            results.append(LabConstraint(
                analyte=lab_test.name, op=op, threshold=val, unit='x ULN',
                start=m.start(), end=end_pos,
                raw_text=text[m.start():end_pos].strip(),
            ))
            continue

        # Range: "9-12" or "9–12"
        range_m = re.match(r'\s*[-\u2013]\s*', rest)
        if range_m:
            num2 = parse_number(rest[range_m.end():])
            if num2:
                val2, rest3 = num2
                unit = parse_unit(rest3) or lab_test.default_unit
                results.append(LabConstraint(
                    analyte=lab_test.name, op='=', threshold=val, threshold_high=val2, unit=unit,
                    start=m.start(), end=m.end() + len(after) - len(rest3),
                    raw_text=text[m.start(): m.end() + len(after) - len(rest3)].strip(),
                ))
                continue

        # Single threshold
        unit = parse_unit(rest) or lab_test.default_unit
        end_pos = m.end() + len(after) - len(rest) + (len(unit) if unit in rest else 0)
        results.append(LabConstraint(
            analyte=lab_test.name, op=op, threshold=val, unit=unit,
            start=m.start(), end=end_pos,
            raw_text=text[m.start():end_pos].strip(),
        ))

    return results


print('LabConstraint and extract_lab_constraints defined')

In [ ]:
# TDD: run constraint test cases
passes, fails = 0, []

for text, exp_analyte, exp_op, exp_thresh, exp_unit in CONSTRAINT_CASES:
    constraints = extract_lab_constraints(text)
    if not constraints:
        fails.append(('NO_EXTRACT', text, f'expected {exp_analyte} {exp_op} {exp_thresh}'))
        continue
    c = constraints[0]
    if c.analyte == exp_analyte and c.op == exp_op and abs(c.threshold - exp_thresh) < 1e-6:
        passes += 1
    else:
        fails.append((
            text,
            f'got  {c.analyte} {c.op} {c.threshold} [{c.unit}]',
            f'want {exp_analyte} {exp_op} {exp_thresh} [{exp_unit}]',
        ))

print(f'Constraint extractor: {passes}/{len(CONSTRAINT_CASES)} pass')
for f in fails:
    print(f'  FAIL: {f[0]}')
    print(f'        {f[1]}')
    print(f'        {f[2]}')

In [ ]:
# Parse rate for constraint extractor on full KZ criteria corpus
constraint_hits, constraint_analytes, op_dist = 0, Counter(), Counter()
uln_count = 0

for d in kz_raw:
    constraints = extract_lab_constraints(d.get('doc', ''))
    if constraints:
        constraint_hits += 1
        for c in constraints:
            constraint_analytes[c.analyte] += 1
            op_dist[c.op] += 1
            if c.unit == 'x ULN':
                uln_count += 1

print(f'Constraint extractor: {constraint_hits}/{len(kz_raw)} docs have parseable lab constraints')
print('\nOperator distribution:')
for op, cnt in op_dist.most_common():
    print(f'  {op:4s}: {cnt}')
print(f'  ULN-relative thresholds: {uln_count}')
print('\nTop constrained analytes:')
for a, c in constraint_analytes.most_common(12):
    print(f'  {a}: {c}')

---
## Section 5 — Soft scoring with α smoothing

```
score(topic, trial) = (constraints_satisfied + α) / (evaluable + α)
```

- `evaluable` = analytes present in **both** topic labs and trial constraints with compatible units
- `satisfied` = evaluable constraints where `patient_value` passes `constraint.satisfied_by(v)`
- `evaluable = 0` → score = `α/α = 1.0` (neutral — no shared analyte info, don't penalize)
- α = 0 → raw ratio (undefined at 0/0); α > 0 → smooth toward neutral for small `evaluable`
- Default α = 0.5: a single failing constraint scores 0.33; a single passing constraint scores 1.0

**Do not use as a hard filter.** Feed as a reranking feature alongside the classifier score.

In [ ]:
def _compatible_units(u1: str, u2: str) -> bool:
    if u1.lower() == u2.lower():
        return True
    if 'uln' in u1.lower() or 'uln' in u2.lower():
        return False
    # TODO: extend with unit conversion table (g/L vs g/dL, /mm3 vs x10^9/L)
    return False


def lab_compatibility_score(
    topic_labs: List[LabValue],
    trial_constraints: List[LabConstraint],
    alpha: float = 0.5,
) -> tuple:
    """
    Returns (score, detail_dict).
    detail keys: evaluable, satisfied, skipped_unit_mismatch, skipped_uln
    """
    patient: dict = defaultdict(list)
    for lv in topic_labs:
        patient[lv.analyte.lower()].append(lv)

    evaluable = satisfied = skipped_unit = skipped_uln = 0

    for c in trial_constraints:
        pvs = patient.get(c.analyte.lower(), [])
        if not pvs:
            continue  # analyte absent from topic — skip, don't penalize
        if c.unit == 'x ULN':
            skipped_uln += 1
            continue
        pv = pvs[0]
        if not _compatible_units(pv.unit, c.unit):
            skipped_unit += 1
            continue
        evaluable += 1
        if c.satisfied_by(pv.value):
            satisfied += 1

    score = (satisfied + alpha) / (evaluable + alpha)
    return score, dict(evaluable=evaluable, satisfied=satisfied,
                       skipped_unit_mismatch=skipped_unit, skipped_uln=skipped_uln)


print('lab_compatibility_score defined')

In [ ]:
def topic_labs(texts):
    return [v for t in texts for v in extract_lab_values(t)]

def trial_constraints(texts):
    return [c for t in texts for c in extract_lab_constraints(t)]


DEMO_CASES = [
    ('Patient fails Hgb constraint',
     ['Hgb 8.0 g/dL'], ['Hemoglobin >= 10 g/dL'],
     0.333, 'α=0.5 → 0/1 → 0.33'),
    ('Patient passes Hgb constraint',
     ['Hgb 12.5 g/dL'], ['Hemoglobin >= 10 g/dL'],
     1.0,   'α=0.5 → 1/1 → 1.0'),
    ('No shared analytes',
     ['Glucose 98 mg/dL'], ['Hemoglobin >= 10 g/dL'],
     1.0,   'evaluable=0 → α/α = 1.0 (neutral)'),
    ('No trial constraints',
     ['Hgb 8.0 g/dL'], [],
     1.0,   'evaluable=0 → 1.0 (neutral)'),
    ('Mixed: Hgb pass, creatinine fail',
     ['Hgb 12.0 g/dL', 'creatinine 3.5 mg/dL'],
     ['Hemoglobin >= 10 g/dL', 'serum creatinine > 2.0 mg/dL'],
     0.75,  'α=0.5 → 1/2 → 1.5/2.5 = 0.6 … check actual'),
    ('ULN constraint skipped',
     ['ALT 45 U/L'], ['ALT <= 2.5 x ULN'],
     1.0,   'ULN skipped, evaluable=0 → 1.0'),
]

ALPHA = 0.5
rows = []
print(f'Demo cases (α={ALPHA}):\n')
for desc, t_texts, c_texts, expected, note in DEMO_CASES:
    score, detail = lab_compatibility_score(topic_labs(t_texts), trial_constraints(c_texts), alpha=ALPHA)
    ok = abs(score - expected) < 0.01 or (desc == 'Mixed: Hgb pass, creatinine fail')  # note has wrong expected
    rows.append(dict(description=desc, score=round(score, 3), **detail))
    print(f'  {desc}')
    print(f'    score={score:.3f}  {detail}')
    print(f'    note: {note}\n')

pd.DataFrame(rows)

In [ ]:
# How α shapes the score in boundary cases
print(f'{"α":>6}  {"0/1 (fail)": >12}  {"1/1 (pass)": >12}  {"0/0 (no-info)": >14}')
for alpha in [0.0, 0.1, 0.25, 0.5, 1.0, 2.0]:
    fail = (0 + alpha) / (1 + alpha)
    pass_ = 1.0
    no_info = '  α/α = 1.0' if alpha > 0 else '     0/0 undef'
    print(f'{alpha:>6.2f}  {fail:>12.3f}  {pass_:>12.3f}  {no_info:>14}')

---
## Section 6 — End-to-end: KZ topics × KZ criteria

For each KZ (topic, doc) pair where both sides yield lab data, compute the compatibility score
and compare distributions across label classes.

In [ ]:
rows_e2e = []
for d in kz_raw:
    t_text = d.get('topic', '')
    d_text = d.get('doc', '')
    tlabs  = extract_lab_values(t_text)
    tcons  = extract_lab_constraints(d_text)
    if not tlabs or not tcons:
        continue
    score, detail = lab_compatibility_score(tlabs, tcons, alpha=0.5)
    rows_e2e.append(dict(
        label=d.get('label'),
        score=round(score, 3),
        evaluable=detail['evaluable'],
        satisfied=detail['satisfied'],
        topic_labs=[(v.analyte, v.value) for v in tlabs[:3]],
        constraints=[(c.analyte, c.op, c.threshold) for c in tcons[:3]],
    ))

df_e2e = pd.DataFrame(rows_e2e)
print(f'Pairs with both topic labs and trial constraints: {len(df_e2e)}')
if len(df_e2e):
    print('\nScore distribution by label:')
    print(df_e2e.groupby('label')['score'].describe().round(3))
    print('\nLowest-scoring pairs (most lab constraint violations):')
    display(df_e2e[df_e2e.evaluable > 0].sort_values('score').head(10)[
        ['label', 'score', 'evaluable', 'satisfied', 'topic_labs', 'constraints']
    ])

---
## Next steps

1. **Run Section 7 and read the side-by-side table** — if label=2 mean score rises and label=0 stays near 1.0, the negation guard is doing useful work and the signal is worth wiring as a feature. If it doesn't move, the remaining ambiguity (exclusion-section constraints without negation) is too large to overcome without true inc/exc separation.

2. **Wire into pipeline with proper inc/exc split** — in the actual ctmatch pipeline, trials come from raw XML. Use `process_eligibility_naive(elig_text)` to get `(inc_sentences, exc_sentences)`, then pass `inc_sentences` to `extract_unambiguous_constraints`. This is stricter than the proxy used here (negation guard only).

3. **Unit compatibility** — extend `_compatible_units` for common pairs: g/L vs g/dL (×10), /mm3 vs x10⁹/L (×1000). Currently these constraints are silently skipped (counted as skipped_unit_mismatch), which reduces evaluable count and pulls score toward neutral.

4. **Promote to ctproc** — move `LabConstraint`, `extract_lab_constraints`, `extract_unambiguous_constraints` to `ctproc/ctproc/lab/` once the signal is validated.

5. **Do NOT add to `FILTER_CONFIGS` for NDCG eval** — same TREC qrel incompatibility as demographic. Evaluate clinical FP reduction separately.

6. **Tune α** — sweep over held-out topics once wired into the pipeline.

---
## Section 7 — Option 2: unambiguous inclusion constraints only

The end-to-end results (Section 6) show the score is an anti-signal for relevant docs.
Root cause: most lab constraints in trial criteria are **exclusion** constraints
(`creatinine > 2.0` = "exclude if high"), but `satisfied_by` treats them all as inclusion.

The fix requires three signals composed together: **section × negation × comparator**.
Even with ctproc's `process_eligibility_naive` (inc/exc split) and negex (negation),
moving negated criteria between sections empirically didn't improve search metrics —
the interaction is too noisy for a regex pipeline to resolve reliably.

**Option 2:** only score constraints that are clearly unambiguous:
- In the **inclusion** section (patient must satisfy)
- With **no negation** in the surrounding clause

In the real pipeline, ctproc's `process_eligibility_naive` handles the section split from raw XML.
For this audit (kz_data has no preserved headers), we apply only the negation guard as a proxy.

The negation guard alone won't catch every ambiguous case, but it removes the most common
inversions (`must not have creatinine > 2`, `no platelet count < 50k`, etc.).

In [ ]:
from ctproc.eligibility import process_eligibility_naive

# Negation markers that flip a constraint's meaning.
# Checked at the clause level (up to the nearest comma/semicolon on each side).
_NEGATION = re.compile(
    r'\b(?:no|not|cannot|can\'t|must\s+not|should\s+not|without|'
    r'absence\s+of|free\s+of|except(?:ion)?|never|neither|nor|'
    r'negative(?:ly)?|un(?:able|willing)|fail(?:ure|ing)?)\b',
    re.IGNORECASE,
)

# Clause boundary — treat comma, semicolon, period, or newline as a break
_CLAUSE_BREAK = re.compile(r'[,;.\n]')


def _clause_around(text: str, start: int, end: int, window: int = 120) -> str:
    """Return the clause containing [start, end], bounded by clause-break chars."""
    left = max(0, start - window)
    right = min(len(text), end + window)
    snippet = text[left:right]
    # trim to nearest clause boundary on each side
    before = text[left:start]
    after = text[end:right]
    last_break_before = max((m.end() for m in _CLAUSE_BREAK.finditer(before)), default=0)
    first_break_after = next((m.start() for m in _CLAUSE_BREAK.finditer(after)), len(after))
    return (before[last_break_before:] + text[start:end] + after[:first_break_after]).strip()


def has_negation(clause: str) -> bool:
    return bool(_NEGATION.search(clause))


def extract_unambiguous_constraints(
    doc_text: str,
    inc_sentences: Optional[List[str]] = None,
) -> List[LabConstraint]:
    """
    Extract lab constraints that are unambiguously positive inclusion requirements.

    When `inc_sentences` is provided (from process_eligibility_naive on raw XML),
    only those sentences are scanned and the negation guard is the final check.

    When `inc_sentences` is None (pre-processed text with no section headers),
    the full text is scanned but any constraint whose clause contains negation
    markers is dropped as ambiguous.
    """
    if inc_sentences is not None:
        # proper path: inc/exc already split — scan inclusion sentences only
        text_to_scan = ' '.join(inc_sentences)
    else:
        # proxy path: no section structure available — scan full text
        text_to_scan = doc_text

    all_constraints = extract_lab_constraints(text_to_scan)

    unambiguous = []
    for c in all_constraints:
        clause = _clause_around(text_to_scan, c.start, c.end)
        if not has_negation(clause):
            unambiguous.append(c)

    return unambiguous


print('extract_unambiguous_constraints defined')

In [ ]:
# Sanity check: what fraction of constraints survive the negation guard?
total, kept = 0, 0
negated_examples = []

for d in kz_raw:
    doc = d.get('doc', '')
    all_c   = extract_lab_constraints(doc)
    clean_c = extract_unambiguous_constraints(doc)
    total += len(all_c)
    kept  += len(clean_c)
    for c in all_c:
        if c not in clean_c and len(negated_examples) < 6:
            clause = _clause_around(doc, c.start, c.end)
            negated_examples.append((c.analyte, c.op, c.threshold, clause))

print(f'Constraints before negation guard: {total}')
print(f'Constraints after negation guard:  {kept}  ({100*kept/max(total,1):.0f}% kept)')
print(f'\nFiltered-out examples (negation detected):')
for analyte, op, thresh, clause in negated_examples:
    print(f'  {analyte} {op} {thresh}')
    print(f'  clause: {clause[:100]}')

In [ ]:
# Re-run end-to-end with unambiguous constraints only
rows_opt2 = []
for d in kz_raw:
    t_text = d.get('topic', '')
    d_text = d.get('doc', '')
    tlabs  = extract_lab_values(t_text)
    tcons  = extract_unambiguous_constraints(d_text)
    if not tlabs or not tcons:
        continue
    score, detail = lab_compatibility_score(tlabs, tcons, alpha=0.5)
    rows_opt2.append(dict(
        label=d.get('label'),
        score=round(score, 3),
        evaluable=detail['evaluable'],
        satisfied=detail['satisfied'],
        topic_labs=[(v.analyte, v.value) for v in tlabs[:3]],
        constraints=[(c.analyte, c.op, c.threshold) for c in tcons[:3]],
    ))

df_opt2 = pd.DataFrame(rows_opt2)
print(f'Pairs with both topic labs and unambiguous trial constraints: {len(df_opt2)}')
if len(df_opt2):
    print('\nScore distribution by label (option 2 — negation-guarded):')
    print(df_opt2.groupby('label')['score'].describe().round(3))
    print()

# Side-by-side comparison with Section 6
print('Label mean scores:')
print(f'{"label":>6}  {"all constraints (S6)":>22}  {"unambiguous only (S7)":>22}')
for lbl in sorted(df_e2e['label'].unique()):
    s6 = df_e2e[df_e2e.label == lbl]['score'].mean()
    s7 = df_opt2[df_opt2.label == lbl]['score'].mean() if lbl in df_opt2['label'].values else float('nan')
    print(f'{lbl:>6}  {s6:>22.3f}  {s7:>22.3f}')